In [ ]:
import json
import pandas as pd

import itables

itables.init_notebook_mode()
itables.options.allow_html = True

import os
os.chdir('/Users/pratuat/Repositories/Factiverse/FactAlign')

## SAFE Results

In [ ]:
sft_safe_v7 = json.load(open("long-form-factuality/results/evals/gemma-2b-sft-v2-2026-06-30-13-20-03-v7-SAFE.json", "r"))
sft_safe_v15 = json.load(open("long-form-factuality/results/evals/gemma-2b-sft-v2-2026-07-02-05-08-34-v15-SAFE.json", "r"))
kto_safe_v7 = json.load(open("long-form-factuality/results/evals/gemma-2b-kto-v2-it_1-2026-06-30-13-34-10-v7-SAFE.json", "r"))
kto_safe_v15 = json.load(open("long-form-factuality/results/evals/gemma-2b-kto-v2-it_1-2026-07-02-05-40-09-v15-SAFE.json", "r"))

# sft_safe_v7.keys()
# dict_keys(['add_universal_postamble', 'max_num_examples', 'num_sentences', 'parallelize', 'responder_model', 'response_length_postamble', 'save_results', 'shared_config', 'show_responder_prompts', 'show_responder_responses', 'shuffle_data', 'side_1', 'side_2', 'task', 'task_short', 'use_length_ablation', 'per_prompt_data', 'total_runtime', 'autoeval_configs', 'side1_avg_num_claims', 'side1_std_num_claims', 'side1_avg_Supported', 'side1_std_Supported', 'side1_avg_Irrelevant', 'side1_std_Irrelevant', 'side1_avg_Not Supported', 'side1_std_Not Supported', 'side1_avg_f1_-1', 'side1_std_f1_-1'])

# sft_safe['per_prompt_data'][0].keys()
# dict_keys(['prompt', 'correct_answers', 'incorrect_answers', 'side1_response', 'side2_response', 'side1_posthoc_eval_data'])

# sft_safe['per_prompt_data'][0]['side1_posthoc_eval_data'].keys()
# dict_keys(['prompt', 'response', 'num_claims', 'sentences_and_atomic_facts', 'all_atomic_facts', 'checked_statements', 'revised_fact_jsonified_all', 'past_steps_jsonified_all', 'Supported', 'Irrelevant', 'Not Supported', 'f1_-1'])

In [ ]:
def collect_safe_result(safe_result):
    data = [result.get('side1_posthoc_eval_data', {}) for result in safe_result.get('per_prompt_data', {})]

    df = pd.DataFrame(data)
    stats = df[['num_claims', 'Supported', 'Not Supported', 'Irrelevant', 'f1_-1']].mean()

    stats['Supported (%)'] = stats['Supported']/stats['num_claims']
    stats['Not Supported (%)'] = stats['Not Supported']/stats['num_claims']
    stats['Irrelevant (%)'] = stats['Irrelevant']/stats['num_claims']

    return df.drop_duplicates(subset=['prompt'], keep="first"), stats

In [ ]:
sft_safe_df, sft_safe_stats = collect_safe_result(sft_safe_v7)
print(sft_safe_stats)

print()
print()

kto_safe_df, kto_safe_stats = collect_safe_result(kto_safe_v7)
print(kto_safe_stats)

In [ ]:
sft_safe_df, sft_safe_stats = collect_safe_result(sft_safe_v15)
print(sft_safe_stats)

print()
print()

kto_safe_df, kto_safe_stats = collect_safe_result(kto_safe_v15)
print(kto_safe_stats)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from html import escape as html_escape

display(HTML("""
<style>
.row-table { table-layout: fixed; width: 100%; margin: 0; border-collapse: collapse; }
.row-table td {
    white-space: pre-wrap; text-align: left; vertical-align: top;
    word-break: break-word; overflow-wrap: break-word; overflow: hidden;
    padding: 4px; border: 1px solid #ddd;
}
</style>
"""))

page_size = 5
cols = ["sft.response", "kto.response"]
display_df = safe_df[cols].reset_index()
display_df["selected"] = False
total_pages = (len(display_df) + page_size - 1) // page_size

def make_page(page):
    page_df = display_df.iloc[page*page_size:(page+1)*page_size]
    row_widgets = []
    for idx, row in page_df.iterrows():
        prompt_val = html_escape(str(row["prompt"])).replace('\n', '<br>') if pd.notna(row["prompt"]) else ''
        sft_val = html_escape(str(row["sft.response"])).replace('\n', '<br>') if pd.notna(row["sft.response"]) else ''
        kto_val = html_escape(str(row["kto.response"])).replace('\n', '<br>') if pd.notna(row["kto.response"]) else ''

        row_html = widgets.HTML(
            value=f'<table class="row-table"><tr>'
                  f'<td style="width:5%">{idx}</td><td style="width:18%">{prompt_val}</td>'
                  f'<td style="width:38%">{sft_val}</td><td style="width:38%">{kto_val}</td>'
                  f'</tr></table>',
            layout=widgets.Layout(width="95%")
        )

        cb = widgets.Checkbox(value=bool(display_df.at[idx, "selected"]), indent=False,
                              layout=widgets.Layout(width="30px", margin="4px 0 0 0"))
        def on_toggle(change, _idx=idx):
            display_df.at[_idx, "selected"] = change["new"]
        cb.observe(on_toggle, names="value")

        row_widgets.append(widgets.HBox([row_html, cb], layout=widgets.Layout(align_items="flex-start")))

    header_html = widgets.HTML(
        value='<table class="row-table" style="font-weight:bold;"><tr>'
              '<td style="width:5%">#</td><td style="width:18%">prompt</td>'
              '<td style="width:38%">sft.response</td><td style="width:38%">kto.response</td>'
              '</tr></table>',
        layout=widgets.Layout(width="95%")
    )
    header_row = widgets.HBox([header_html, widgets.HTML(value="<b>sel</b>", layout=widgets.Layout(width="30px"))])
    return widgets.VBox([header_row] + row_widgets)

output = widgets.Output()
page_slider = widgets.IntSlider(value=0, min=0, max=total_pages - 1, description="Page:")
page_label = widgets.Label(value=f"Page 1 / {total_pages}")

def on_page_change(change):
    page_label.value = f"Page {change['new'] + 1} / {total_pages}"
    output.clear_output(wait=True)
    with output:
        display(make_page(change['new']))

page_slider.observe(on_page_change, names='value')
with output:
    display(make_page(0))

widgets.VBox([widgets.HBox([page_slider, page_label]), output])

## Long-form Response

In [ ]:
sft_rsp_file = "../long-form-factuality/results/evals/gemma-2b-sft-v2-2026-06-30-13-20-03.json"
sft_rsp = json.load(open(sft_rsp_file, "r"))


kto_rsp_file = "../long-form-factuality/results/evals/gemma-2b-kto-v2-it_1-2026-06-30-13-34-10.json"
kto_rsp = json.load(open(kto_rsp_file, "r"))

In [ ]:
def collect_responses(responses):
    data = [
        {
            "prompt": response['prompt'],
            "side1_response": response.get('side1_response'),
            "side2_response": response.get('side2_response'),
        } for response in responses['per_prompt_data']
    ]

    return pd.DataFrame(data).set_index('prompt').replace(to_replace='', value=None).dropna(how='all', axis=1)

In [ ]:
sft_df = collect_responses(sft_result)
kto_df = collect_responses(kto_result)

In [ ]:
df = pd.concat([sft_df.add_prefix('sft.'), kto_df.add_prefix('kto.')], axis=1)

In [ ]:
Who is Albert Einstine? Provide as many specific details and examples as possible (such as names of people, numbers, events, locations, dates, times, etc.)

In [ ]:
df.index

In [ ]:
for prompt, row in df.iterrows():
    print('='*100)
    print(f"[Prompt]: {prompt}")
    print('-'*100)
    print(f"[SFT]: {row['sft.side1_response']}")
    print('-'*100)
    print(f"[KTO]: {row['kto.side1_response']}")
    print('-'*100)


## SAFE Evaluation

In [ ]:
def show_stats(result):
    print(result['responder_model'])
    print(f"{'':-^100}")
    print(f"Average no. of claims: {result['side1_avg_num_claims']} \t Std: {result['side1_std_num_claims']}", )
    print(f"Average no. of SUPPORTED claims: {result['side1_avg_Supported']} \t Std: {result['side1_std_Supported']}", )
    print(f"Average no. of IRRELEVANT claims: {result['side1_avg_Irrelevant']} \t Std: {result['side1_std_Irrelevant']}", )
    print(f"Average no. of NOT-SUPPORTED claims: {result['side1_avg_Not Supported']} \t Std: {result['side1_std_Not Supported']}", )
    print(f"F1 - 1: {result['side1_avg_f1_-1']} \t Std: {result['side1_std_f1_-1']}", )

show_stats(baseline_result)

print()
print()

show_stats(challenger_result)


In [ ]:
baseline_result['per_prompt_data'][1]['side1_response']

In [ ]:
print(baseline_result['per_prompt_data'][1].keys())

print(baseline_result['per_prompt_data'][1]['side1_posthoc_eval_data'].keys())

In [ ]:
per_prompt_data = []
for el in baseline_result['per_prompt_data']:
    per_prompt_data.append({
        "num_of_claims": el['side1_posthoc_eval_data']['num_claims'],
        "supported": el['side1_posthoc_eval_data']['Supported'],
        "irrelevant": el['side1_posthoc_eval_data']['Irrelevant'],
        "not_supported": el['side1_posthoc_eval_data']['Not Supported'],
    })

df = pd.DataFrame(per_prompt_data)


In [ ]:
i = 64

print(baseline_result['per_prompt_data'][i]['prompt'])
print(baseline_result['per_prompt_data'][i]['side1_response'])